# Zaid Hybrid IDS — Step 11B: Quantization and Full-Pipeline Optimization

This notebook tests whether the validated Qwen2.5-0.5B student can meet the thesis efficiency targets without weakening explanation quality.

Scientific safeguards:

- Validation only: the test file is not defined, opened, or evaluated.
- The frozen student, prompt, deterministic generation settings, and strict evidence checks are preserved.
- FP16, INT8, and NF4 are compared on the same records and hardware.
- Batch throughput is reported separately from true single-alert latency.
- The thesis `<50 ms` claim is judged from single-alert P95 full-pipeline latency, not batch-normalized throughput.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)


Mounted at /content/drive


In [3]:
!pip install -q "transformers>=5.0.0,<6.0.0" "huggingface-hub>=1.16.0,<2.0" accelerate bitsandbytes sentencepiece safetensors pandas numpy rouge-score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.6 MB/s eta 0:00:00


In [4]:
from pathlib import Path
from time import perf_counter
from collections import Counter
import gc
import hashlib
import json
import math
import platform
import re
import statistics

import numpy as np
import pandas as pd
import torch
import transformers
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 20260831
torch.manual_seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
VALIDATION_PATH = PROJECT_ROOT / 'data/explainer/qwen25_15b_strict_repair_v1/teacher_targets_validation.jsonl'
STUDENT_CANDIDATES = [
    PROJECT_ROOT / 'models/student_qwen25_05b_distilled_v1_validated/best_model',
    PROJECT_ROOT / 'models/student_qwen25_05b_distilled_v1/best_model',
    PROJECT_ROOT / 'models/student_qwen25_05b_distilled_v1/checkpoints/checkpoint-140',
]
STUDENT_DIR = next((p for p in STUDENT_CANDIDATES if p.exists()), None)
RESULT_DIR = PROJECT_ROOT / 'results/quantization_full_pipeline_v1'

assert VALIDATION_PATH.exists(), f'Missing validation file: {VALIDATION_PATH}'
assert STUDENT_DIR is not None, 'No validated student model folder was found.'
assert torch.cuda.is_available(), 'A GPU runtime is required. Select T4 GPU and rerun.'

RESULT_DIR.mkdir(parents=True, exist_ok=True)
device_name = torch.cuda.get_device_name(0)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with VALIDATION_PATH.open('r', encoding='utf-8') as handle:
    validation_records = [json.loads(line) for line in handle if line.strip()]

assert len(validation_records) == 120, len(validation_records)
assert all(r.get('split') == 'validation' for r in validation_records)
assert len({r['event_id'] for r in validation_records}) == len(validation_records)

print('Student directory:', STUDENT_DIR)
print('Validation events:', len(validation_records))
print('Validation labels:', dict(Counter(r.get('research_original_label') for r in validation_records)))
print('Validation SHA-256:', sha256_file(VALIDATION_PATH))
print('Transformers:', transformers.__version__)
print('PyTorch:', torch.__version__)
print('GPU:', device_name)
print('Test records loaded: 0')
print('QUANTIZATION OPTIMIZATION SETUP: READY')


Student directory: /content/drive/MyDrive/Zaid_Hybrid_IDS/models/student_qwen25_05b_distilled_v1_validated/best_model
Validation events: 120
Validation labels: {'FTP-BruteForce': 60, 'SSH-Bruteforce': 60}
Validation SHA-256: 44ee031e1469ba6f86f2a5efb2545f5fd31839f3d830b942df098cacf2466cca
Transformers: 5.15.1
PyTorch: 2.11.0+cu128
GPU: Tesla T4
Test records loaded: 0
QUANTIZATION OPTIMIZATION SETUP: READY


In [5]:
SYSTEM_INSTRUCTION = '''You are a defensive intrusion-detection analyst. Follow the requested output schema exactly. Use only supplied detector and SHAP evidence. Do not infer a specific attack subtype, identity, endpoint, port, cause, or offensive procedure. SHAP is local model evidence and not causal proof. Return no preamble, bullets, markdown, or text after the required three lines.'''

def direction_requirement(first, second):
    first_supports = first['direction'] == 'supports_attack'
    second_supports = second['direction'] == 'supports_attack'
    if first_supports and second_supports:
        return (
            f"State that {first['name']} and {second['name']} provided the "
            "strongest local SHAP support for the detector decision."
        )
    if first_supports and not second_supports:
        return (
            f"State that {first['name']} supported the detector decision, while "
            f"{second['name']} opposed it."
        )
    if not first_supports and second_supports:
        return (
            f"State that {second['name']} supported the detector decision, while "
            f"{first['name']} opposed it."
        )
    return (
        f"State that {first['name']} and {second['name']} locally opposed the "
        "detector decision."
    )

def build_prompt(record):
    first, second = record['top_features'][:2]
    exact_confidence = f"{record['confidence']:.3f}"
    relation = direction_requirement(first, second)
    return f'''Write exactly three short lines.
Line 1 must begin Summary:. Line 2 must begin Evidence:. Line 3 must begin Action:.
Summary must contain the decision Attack and the exact confidence {exact_confidence}.
Evidence must copy these names exactly: {first['name']} and {second['name']}.
Do not mention any other feature name or any attack subtype.

Detector decision: Attack
Confidence: {exact_confidence}
Feature 1: {first['name']} (value={first['value']:.6g}, SHAP={first['shap_value']:+.6f}, direction={first['direction']})
Feature 2: {second['name']} (value={second['value']:.6g}, SHAP={second['shap_value']:+.6f}, direction={second['direction']})

Required evidence meaning: {relation}
Required action: advise reviewing network, authentication, and host logs and corroborating before containment.'''

def target_text(record):
    return (
        record.get('target_text')
        or record.get('final_target')
        or record.get('target')
        or ''
    ).strip()

IP_PATTERN = re.compile(r'(?<![\d.])(?:\d{1,3}\.){3}\d{1,3}(?![\d.])')
PROHIBITED_PATTERN = re.compile(
    r'(?i)\b(exploit|payload|reverse shell|credential stuffing tool|malware family|attacker identity)\b'
)
ATTACK_SUBTYPE_PATTERN = re.compile(
    r'(?i)\b(ftp|ssh|brute[ -]?force|botnet|ddos|dos|infiltration|sql injection|xss|heartbleed)\b'
)
KNOWN_FEATURE_NAMES = sorted({
    feature['name']
    for record in validation_records
    for feature in record['top_features']
}, key=len, reverse=True)

def normalize_output(text):
    text = text.strip().replace('\r\n', '\n').replace('\r', '\n')
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return '\n'.join(lines)

def strict_validate(record, text):
    normalized = normalize_output(text)
    lines = normalized.splitlines()
    schema = (
        len(lines) == 3
        and lines[0].startswith('Summary:')
        and lines[1].startswith('Evidence:')
        and lines[2].startswith('Action:')
    )
    first, second = record['top_features'][:2]
    summary = lines[0] if len(lines) > 0 else ''
    evidence = lines[1] if len(lines) > 1 else ''
    action = lines[2] if len(lines) > 2 else ''
    exact_confidence = f"{record['confidence']:.3f}"
    decision_aligned = bool(re.search(r'(?i)\battack\b', summary))
    confidence_aligned = exact_confidence in summary
    required_features = first['name'] in evidence and second['name'] in evidence
    allowed_names = {first['name'], second['name']}
    unsupported = any(
        name in evidence and name not in allowed_names
        for name in KNOWN_FEATURE_NAMES
    )
    directions = {first['direction'], second['direction']}
    evidence_lower = evidence.lower()
    if directions == {'supports_attack'}:
        direction_aligned = 'support' in evidence_lower
    elif directions == {'opposes_attack'}:
        direction_aligned = 'oppos' in evidence_lower
    else:
        direction_aligned = 'support' in evidence_lower and 'oppos' in evidence_lower
    action_lower = action.lower()
    action_grounded = (
        'review' in action_lower
        and 'log' in action_lower
        and 'corroborat' in action_lower
        and 'containment' in action_lower
    )
    prohibited = bool(
        IP_PATTERN.search(normalized)
        or PROHIBITED_PATTERN.search(normalized)
        or ATTACK_SUBTYPE_PATTERN.search(normalized)
    )
    valid = all([
        schema, decision_aligned, confidence_aligned, required_features,
        direction_aligned, action_grounded, not unsupported, not prohibited,
    ])
    return {
        'normalized': normalized,
        'schema_compliant': schema,
        'decision_aligned': decision_aligned,
        'confidence_aligned': confidence_aligned,
        'required_features_present': required_features,
        'direction_aligned': direction_aligned,
        'action_grounded': action_grounded,
        'unsupported_feature': unsupported,
        'prohibited_content': prohibited,
        'valid': valid,
    }

tokenizer = AutoTokenizer.from_pretrained(STUDENT_DIR, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_prompt(record):
    return tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': SYSTEM_INSTRUCTION},
            {'role': 'user', 'content': build_prompt(record)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

validation_prompts = [render_prompt(r) for r in validation_records]
assert all(target_text(r) for r in validation_records)
print('Maximum prompt tokens:', max(len(tokenizer.encode(p, add_special_tokens=False)) for p in validation_prompts))
print('Prompt and validator: FROZEN')


Maximum prompt tokens: 298
Prompt and validator: FROZEN


In [6]:
GENERATION_CONFIG = {
    'max_new_tokens': 128,
    'do_sample': False,
    'repetition_penalty': 1.05,
    'use_cache': True,
}

# Existing measured Stage-1 components retained for the full-pipeline calculation.
DETECTOR_MS_PER_FLOW = 0.1560
SHAP_MS_PER_ALERT = 1.7696
THESIS_DETECTION_TARGET_MS = 10.0
THESIS_FULL_PIPELINE_TARGET_MS = 50.0

PILOT_COUNT = 24
PILOT_RECORDS = validation_records[:PILOT_COUNT]
PILOT_PROMPTS = validation_prompts[:PILOT_COUNT]

VARIANTS = [
    {'name': 'fp16_sdpa', 'kind': 'fp16'},
    {'name': 'int8_bnb', 'kind': 'int8'},
    {'name': 'nf4_bnb', 'kind': 'nf4'},
]

def unload_model(model):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def load_variant(kind):
    common = dict(
        pretrained_model_name_or_path=STUDENT_DIR,
        trust_remote_code=True,
        device_map='auto',
        attn_implementation='sdpa',
    )
    if kind == 'fp16':
        return AutoModelForCausalLM.from_pretrained(
            **common, dtype=torch.float16
        ).eval()
    if kind == 'int8':
        config = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            **common, quantization_config=config, dtype=torch.float16
        ).eval()
    if kind == 'nf4':
        config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        return AutoModelForCausalLM.from_pretrained(
            **common, quantization_config=config, dtype=torch.float16
        ).eval()
    raise ValueError(kind)

def quantile(values, q):
    return float(np.quantile(np.asarray(values, dtype=float), q, method='linear'))

def run_benchmark(model, records, prompts, batch_size, warmup_batches=2):
    # Warmup is excluded and uses real prompts.
    for start in range(0, min(len(prompts), batch_size * warmup_batches), batch_size):
        batch = prompts[start:start + batch_size]
        encoded = tokenizer(
            batch, return_tensors='pt', padding=True, truncation=True, max_length=512
        ).to('cuda')
        with torch.inference_mode():
            model.generate(
                **encoded,
                **GENERATION_CONFIG,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    batch_wall_ms = []
    event_effective_ms = []
    outputs = []
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        actual_batch = len(batch_prompts)
        torch.cuda.synchronize()
        start_time = perf_counter()
        encoded = tokenizer(
            batch_prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        ).to('cuda')
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                **GENERATION_CONFIG,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        continuation = generated[:, encoded['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(continuation, skip_special_tokens=True)
        torch.cuda.synchronize()
        elapsed_ms = (perf_counter() - start_time) * 1000.0
        batch_wall_ms.append(elapsed_ms)
        event_effective_ms.extend([elapsed_ms / actual_batch] * actual_batch)
        outputs.extend(decoded)

    checks = [strict_validate(record, output) for record, output in zip(records, outputs)]
    reference_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = [
        reference_scorer.score(target_text(record), check['normalized'])['rougeL'].fmeasure
        for record, check in zip(records, checks)
    ]
    generated_tokens = [
        len(tokenizer.encode(check['normalized'], add_special_tokens=False))
        for check in checks
    ]
    return {
        'outputs': outputs,
        'checks': checks,
        'event_effective_ms': event_effective_ms,
        'batch_wall_ms': batch_wall_ms,
        'generated_tokens': generated_tokens,
        'rouge_l': rouge_l,
        'peak_gpu_mb': torch.cuda.max_memory_allocated() / (1024 ** 2),
    }

print('Pilot events:', PILOT_COUNT)
print('Generation configuration:', GENERATION_CONFIG)
print('Detector ms/flow:', DETECTOR_MS_PER_FLOW)
print('SHAP ms/alert:', SHAP_MS_PER_ALERT)
print('QUANTIZATION BENCHMARK: READY')


Pilot events: 24
Generation configuration: {'max_new_tokens': 128, 'do_sample': False, 'repetition_penalty': 1.05, 'use_cache': True}
Detector ms/flow: 0.156
SHAP ms/alert: 1.7696
QUANTIZATION BENCHMARK: READY


In [7]:
pilot_rows = []
pilot_outputs = {}

for variant in VARIANTS:
    model = None
    print('=' * 80)
    print('Loading variant:', variant['name'])
    try:
        model = load_variant(variant['kind'])
        footprint_mb = model.get_memory_footprint() / (1024 ** 2)
        for batch_size in [1, 4, 8]:
            result = run_benchmark(
                model, PILOT_RECORDS, PILOT_PROMPTS, batch_size=batch_size
            )
            valid_count = sum(check['valid'] for check in result['checks'])
            mean_effective = statistics.mean(result['event_effective_ms'])
            p95_effective = quantile(result['event_effective_ms'], 0.95)
            p99_effective = quantile(result['event_effective_ms'], 0.99)
            mean_batch_wall = statistics.mean(result['batch_wall_ms'])
            full_pipeline_mean = mean_effective + DETECTOR_MS_PER_FLOW + SHAP_MS_PER_ALERT
            full_pipeline_p95 = p95_effective + DETECTOR_MS_PER_FLOW + SHAP_MS_PER_ALERT
            row = {
                'scope': 'pilot_validation_24',
                'variant': variant['name'],
                'quantization': variant['kind'],
                'batch_size': batch_size,
                'events': len(PILOT_RECORDS),
                'valid_count': valid_count,
                'valid_rate': valid_count / len(PILOT_RECORDS),
                'mean_generation_effective_ms_per_event': mean_effective,
                'p95_generation_effective_ms_per_event': p95_effective,
                'p99_generation_effective_ms_per_event': p99_effective,
                'mean_batch_wall_ms': mean_batch_wall,
                'throughput_events_per_second': 1000.0 / mean_effective,
                'full_pipeline_mean_ms_per_event': full_pipeline_mean,
                'full_pipeline_p95_ms_per_event': full_pipeline_p95,
                'mean_generated_tokens': statistics.mean(result['generated_tokens']),
                'mean_rouge_l': statistics.mean(result['rouge_l']),
                'model_footprint_mb': footprint_mb,
                'peak_gpu_allocated_mb': result['peak_gpu_mb'],
                'meets_50ms_p95': full_pipeline_p95 < THESIS_FULL_PIPELINE_TARGET_MS,
                'error': None,
            }
            pilot_rows.append(row)
            pilot_outputs[(variant['name'], batch_size)] = result
            print(
                variant['name'], 'batch', batch_size,
                '| valid', f'{valid_count}/{len(PILOT_RECORDS)}',
                '| effective mean ms/event', round(mean_effective, 3),
                '| p95 full pipeline ms', round(full_pipeline_p95, 3),
                '| footprint MB', round(footprint_mb, 1),
            )
    except Exception as exc:
        pilot_rows.append({
            'scope': 'pilot_validation_24',
            'variant': variant['name'],
            'quantization': variant['kind'],
            'batch_size': None,
            'events': 0,
            'valid_count': 0,
            'valid_rate': 0.0,
            'error': f'{type(exc).__name__}: {exc}',
        })
        print('Variant failed safely:', type(exc).__name__, exc)
    finally:
        unload_model(model)

pilot_df = pd.DataFrame(pilot_rows)
display(pilot_df)
print('Test records accessed: 0')
print('QUANTIZATION PILOT: COMPLETE')


Loading variant: fp16_sdpa


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

fp16_sdpa batch 1 | valid 24/24 | effective mean ms/event 2563.402 | p95 full pipeline ms 3309.22 | footprint MB 942.3
fp16_sdpa batch 4 | valid 24/24 | effective mean ms/event 700.938 | p95 full pipeline ms 840.11 | footprint MB 942.3
fp16_sdpa batch 8 | valid 24/24 | effective mean ms/event 374.277 | p95 full pipeline ms 434.566 | footprint MB 942.3
Loading variant: int8_bnb


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

int8_bnb batch 1 | valid 24/24 | effective mean ms/event 10791.38 | p95 full pipeline ms 11797.622 | footprint MB 601.0
int8_bnb batch 4 | valid 24/24 | effective mean ms/event 3194.698 | p95 full pipeline ms 3227.035 | footprint MB 601.0
int8_bnb batch 8 | valid 24/24 | effective mean ms/event 1607.06 | p95 full pipeline ms 1626.551 | footprint MB 601.0
Loading variant: nf4_bnb


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

nf4_bnb batch 1 | valid 24/24 | effective mean ms/event 3791.342 | p95 full pipeline ms 4688.994 | footprint MB 430.4
nf4_bnb batch 4 | valid 24/24 | effective mean ms/event 980.78 | p95 full pipeline ms 1067.139 | footprint MB 430.4
nf4_bnb batch 8 | valid 24/24 | effective mean ms/event 526.536 | p95 full pipeline ms 603.594 | footprint MB 430.4


,scope,variant,quantization,batch_size,events,valid_count,valid_rate,mean_generation_effective_ms_per_event,p95_generation_effective_ms_per_event,p99_generation_effective_ms_per_event,mean_batch_wall_ms,throughput_events_per_second,full_pipeline_mean_ms_per_event,full_pipeline_p95_ms_per_event,mean_generated_tokens,mean_rouge_l,model_footprint_mb,peak_gpu_allocated_mb,meets_50ms_p95,error
0,pilot_validation_24,fp16_sdpa,fp16,1,24,24,1.0,2563.401754,3307.294139,3357.227054,2563.401754,0.390107,2565.327354,3309.219739,80.166667,1.0,942.292969,972.540039,False,None
1,pilot_validation_24,fp16_sdpa,fp16,4,24,24,1.0,700.938365,838.183931,838.183931,2803.753461,1.426659,702.863965,840.109531,80.166667,1.0,942.292969,1034.174805,False,None
2,pilot_validation_24,fp16_sdpa,fp16,8,24,24,1.0,374.276967,432.640429,432.640429,2994.215738,2.671818,376.202567,434.566029,80.166667,1.0,942.292969,1116.809570,False,None
3,pilot_validation_24,int8_bnb,int8,1,24,24,1.0,10791.380315,11795.696530,11832.257112,10791.380315,0.092667,10793.305915,11797.622130,80.166667,1.0,601.042969,632.541504,False,None
4,pilot_validation_24,int8_bnb,int8,4,24,24,1.0,3194.697848,3225.108919,3225.108919,12778.791394,0.313019,3196.623448,3227.034519,80.166667,1.0,601.042969,694.178223,False,None
5,pilot_validation_24,int8_bnb,int8,8,24,24,1.0,1607.059724,1624.625119,1624.625119,12856.477789,0.622254,1608.985324,1626.550719,80.166667,1.0,601.042969,776.571777,False,None
6,pilot_validation_24,nf4_bnb,nf4,1,24,24,1.0,3791.341755,4687.067995,4699.788649,3791.341755,0.263759,3793.267355,4688.993595,80.166667,1.0,430.417969,466.454102,False,None
7,pilot_validation_24,nf4_bnb,nf4,4,24,24,1.0,980.779860,1065.213856,1065.213856,3923.119440,1.019597,982.705460,1067.139456,80.166667,1.0,430.417969,528.280273,False,None
8,pilot_validation_24,nf4_bnb,nf4,8,24,24,1.0,526.535618,601.668537,601.668537,4212.284944,1.899207,528.461218,603.594137,80.166667,1.0,430.417969,610.286133,False,None


Test records accessed: 0
QUANTIZATION PILOT: COMPLETE


In [8]:
# A variant is eligible only if it passes all strict checks on all 24 records.
eligible = pilot_df[
    (pilot_df['events'] == PILOT_COUNT)
    & (pilot_df['valid_rate'] == 1.0)
    & pilot_df['error'].isna()
].copy()
assert not eligible.empty, 'No quantization variant preserved strict validity.'

# Select a genuinely quantized deployment candidate from batch=1 because this
# is the thesis latency criterion. FP16 remains the scientific baseline only.
eligible_single = eligible[eligible['batch_size'] == 1].copy()
assert not eligible_single.empty, 'No strictly valid single-alert variant.'
eligible_quantized = eligible_single[
    eligible_single['quantization'].isin(['int8', 'nf4'])
].copy()
assert not eligible_quantized.empty, (
    'Neither INT8 nor NF4 preserved strict validity. Do not claim quantization.'
)
best_row = eligible_quantized.sort_values(
    ['full_pipeline_p95_ms_per_event', 'model_footprint_mb']
).iloc[0]
BEST_VARIANT = str(best_row['variant'])
BEST_KIND = str(best_row['quantization'])

print('Selected variant:', BEST_VARIANT)
print('Selected quantization:', BEST_KIND)
print('FP16 is used as baseline only: True')
print('Pilot strict validity:', best_row['valid_rate'])
print('Pilot single-alert P95 full pipeline ms:', best_row['full_pipeline_p95_ms_per_event'])
print('Pilot meets <50 ms:', bool(best_row['meets_50ms_p95']))
print('BEST VALID VARIANT: SELECTED')


Selected variant: nf4_bnb
Selected quantization: nf4
FP16 is used as baseline only: True
Pilot strict validity: 1.0
Pilot single-alert P95 full pipeline ms: 4688.993594799837
Pilot meets <50 ms: False
BEST VALID VARIANT: SELECTED


In [9]:
best_model = load_variant(BEST_KIND)
full_rows = []
full_results = {}

for batch_size in [1, 4, 8, 16]:
    print('Running all validation events with batch size:', batch_size)
    result = run_benchmark(
        best_model, validation_records, validation_prompts, batch_size=batch_size
    )
    full_results[batch_size] = result
    valid_count = sum(check['valid'] for check in result['checks'])
    mean_effective = statistics.mean(result['event_effective_ms'])
    p50_effective = quantile(result['event_effective_ms'], 0.50)
    p95_effective = quantile(result['event_effective_ms'], 0.95)
    p99_effective = quantile(result['event_effective_ms'], 0.99)
    full_rows.append({
        'scope': 'full_validation_120',
        'variant': BEST_VARIANT,
        'quantization': BEST_KIND,
        'batch_size': batch_size,
        'events': len(validation_records),
        'valid_count': valid_count,
        'valid_rate': valid_count / len(validation_records),
        'fallback_rate': 1.0 - valid_count / len(validation_records),
        'mean_generation_effective_ms_per_event': mean_effective,
        'p50_generation_effective_ms_per_event': p50_effective,
        'p95_generation_effective_ms_per_event': p95_effective,
        'p99_generation_effective_ms_per_event': p99_effective,
        'mean_batch_wall_ms': statistics.mean(result['batch_wall_ms']),
        'throughput_events_per_second': 1000.0 / mean_effective,
        'full_pipeline_mean_ms_per_event': mean_effective + DETECTOR_MS_PER_FLOW + SHAP_MS_PER_ALERT,
        'full_pipeline_p95_ms_per_event': p95_effective + DETECTOR_MS_PER_FLOW + SHAP_MS_PER_ALERT,
        'mean_generated_tokens': statistics.mean(result['generated_tokens']),
        'mean_rouge_l': statistics.mean(result['rouge_l']),
        'peak_gpu_allocated_mb': result['peak_gpu_mb'],
        'meets_50ms_p95': p95_effective + DETECTOR_MS_PER_FLOW + SHAP_MS_PER_ALERT < THESIS_FULL_PIPELINE_TARGET_MS,
    })
    print('Completed:', batch_size, '| valid:', f'{valid_count}/120')

full_df = pd.DataFrame(full_rows)
display(full_df)
print('Test records accessed: 0')
print('FULL VALIDATION SPEED BENCHMARK: COMPLETE')


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running all validation events with batch size: 1
Completed: 1 | valid: 120/120
Running all validation events with batch size: 4
Completed: 4 | valid: 120/120
Running all validation events with batch size: 8
Completed: 8 | valid: 120/120
Running all validation events with batch size: 16
Completed: 16 | valid: 120/120


,scope,variant,quantization,batch_size,events,valid_count,valid_rate,fallback_rate,mean_generation_effective_ms_per_event,p50_generation_effective_ms_per_event,p95_generation_effective_ms_per_event,p99_generation_effective_ms_per_event,mean_batch_wall_ms,throughput_events_per_second,full_pipeline_mean_ms_per_event,full_pipeline_p95_ms_per_event,mean_generated_tokens,mean_rouge_l,peak_gpu_allocated_mb,meets_50ms_p95
0,full_validation_120,nf4_bnb,nf4,1,120,120,1.0,0.0,3719.309747,3749.606099,4684.400333,4775.371211,3719.309747,0.268867,3721.235347,4686.325933,79.1,0.977182,902.833008,False
1,full_validation_120,nf4_bnb,nf4,4,120,120,1.0,0.0,957.334973,932.348451,1156.959261,1174.446335,3829.339894,1.044566,959.260573,1158.884861,79.1,0.977182,964.958984,False
2,full_validation_120,nf4_bnb,nf4,8,120,120,1.0,0.0,501.629349,484.878688,605.132154,605.132154,4013.034790,1.993504,503.554949,607.057754,79.1,0.977182,1046.570312,False
3,full_validation_120,nf4_bnb,nf4,16,120,120,1.0,0.0,275.866967,256.276774,459.933537,459.933537,4138.004508,3.624936,277.792567,461.859137,79.1,0.977182,1213.148926,False


Test records accessed: 0
FULL VALIDATION SPEED BENCHMARK: COMPLETE


In [10]:
single = full_df.loc[full_df['batch_size'] == 1].iloc[0]
all_quality_pass = bool((full_df['valid_rate'] == 1.0).all())
detection_pass = DETECTOR_MS_PER_FLOW < THESIS_DETECTION_TARGET_MS
full_pipeline_pass = bool(single['meets_50ms_p95'])

QUANTIZED_MODEL_DIR = (
    PROJECT_ROOT
    / 'models/student_qwen25_05b_distilled_quantized_v1'
    / BEST_VARIANT
)
quantized_model_saved = False
quantized_model_save_error = None
try:
    QUANTIZED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    best_model.save_pretrained(QUANTIZED_MODEL_DIR, safe_serialization=True)
    tokenizer.save_pretrained(QUANTIZED_MODEL_DIR)
    quantized_model_saved = True
except Exception as exc:
    quantized_model_save_error = f'{type(exc).__name__}: {exc}'

serialized_model_bytes = (
    sum(path.stat().st_size for path in QUANTIZED_MODEL_DIR.rglob('*') if path.is_file())
    if quantized_model_saved else None
)

summary = {
    'schema_version': '1.0',
    'experiment': 'student_quantization_full_pipeline_optimization',
    'seed': SEED,
    'date_utc': pd.Timestamp.utcnow().isoformat(),
    'scope': 'validation_only',
    'test_records_accessed': 0,
    'validation_events': len(validation_records),
    'validation_sha256': sha256_file(VALIDATION_PATH),
    'student_dir': str(STUDENT_DIR),
    'gpu': device_name,
    'transformers_version': transformers.__version__,
    'torch_version': torch.__version__,
    'generation_config': GENERATION_CONFIG,
    'selected_variant': BEST_VARIANT,
    'selected_quantization': BEST_KIND,
    'quantized_model_saved': quantized_model_saved,
    'quantized_model_dir': str(QUANTIZED_MODEL_DIR),
    'quantized_model_save_error': quantized_model_save_error,
    'serialized_quantized_model_mb': (
        serialized_model_bytes / (1024 ** 2)
        if serialized_model_bytes is not None else None
    ),
    'detector_ms_per_flow': DETECTOR_MS_PER_FLOW,
    'shap_ms_per_alert': SHAP_MS_PER_ALERT,
    'thesis_detection_target_ms': THESIS_DETECTION_TARGET_MS,
    'thesis_full_pipeline_target_ms': THESIS_FULL_PIPELINE_TARGET_MS,
    'quality_pass_120': all_quality_pass,
    'detection_target_pass': detection_pass,
    'single_alert_p95_full_pipeline_ms': float(single['full_pipeline_p95_ms_per_event']),
    'full_pipeline_50ms_pass': full_pipeline_pass,
    'all_numeric_targets_pass': bool(
        all_quality_pass
        and detection_pass
        and full_pipeline_pass
        and quantized_model_saved
    ),
}

pilot_path = RESULT_DIR / 'quantization_pilot_24.csv'
full_path = RESULT_DIR / 'quantization_full_validation_120.csv'
manifest_path = RESULT_DIR / 'quantization_full_pipeline_manifest.json'
outputs_path = RESULT_DIR / 'selected_variant_validation_outputs.jsonl'

pilot_df.to_csv(pilot_path, index=False)
full_df.to_csv(full_path, index=False)
manifest_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
with outputs_path.open('w', encoding='utf-8') as handle:
    selected_outputs = full_results[1]['outputs']
    selected_checks = full_results[1]['checks']
    for record, output, check in zip(validation_records, selected_outputs, selected_checks):
        handle.write(json.dumps({
            'event_id': record['event_id'],
            'split': 'validation',
            'variant': BEST_VARIANT,
            'output': output,
            'normalized_output': check['normalized'],
            'strict_valid': check['valid'],
        }, ensure_ascii=False) + '\n')

print('Quality pass on 120 Validation records:', all_quality_pass)
print('Detection target <10 ms:', detection_pass)
print('Single-alert P95 full pipeline ms:', summary['single_alert_p95_full_pipeline_ms'])
print('Full pipeline target <50 ms:', full_pipeline_pass)
print('Quantized model saved:', quantized_model_saved)
print('Quantized model directory:', QUANTIZED_MODEL_DIR)
if quantized_model_save_error:
    print('Quantized model save error:', quantized_model_save_error)
print('ALL NUMERIC THESIS TARGETS PASS:', summary['all_numeric_targets_pass'])
print('Saved:', pilot_path)
print('Saved:', full_path)
print('Saved:', manifest_path)
print('Saved:', outputs_path)
print('QUANTIZATION AND FULL-PIPELINE OPTIMIZATION: SAVED')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Quality pass on 120 Validation records: True
Detection target <10 ms: True
Single-alert P95 full pipeline ms: 4686.325932600136
Full pipeline target <50 ms: False
Quantized model saved: True
Quantized model directory: /content/drive/MyDrive/Zaid_Hybrid_IDS/models/student_qwen25_05b_distilled_quantized_v1/nf4_bnb
ALL NUMERIC THESIS TARGETS PASS: False
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/quantization_full_pipeline_v1/quantization_pilot_24.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/quantization_full_pipeline_v1/quantization_full_validation_120.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/quantization_full_pipeline_v1/quantization_full_pipeline_manifest.json
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/quantization_full_pipeline_v1/selected_variant_validation_outputs.jsonl
QUANTIZATION AND FULL-PIPELINE OPTIMIZATION: SAVED


## Interpretation rule

- If `ALL NUMERIC THESIS TARGETS PASS: True`, the current validated configuration supports all three numerical thesis targets on the specified Tesla T4 validation protocol.
- If it is `False`, do not relabel batch-normalized throughput as single-alert latency. Preserve these results as the honest quantization experiment and proceed to an optimized inference engine or revise the `<50 ms` proposal target to an evidence-based threshold.
